Import Dependencies


In [62]:
import os
import openai
from qdrant_client import QdrantClient
from langsmith import Client

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

Download an example reference data point from langsmith

In [63]:
from dotenv import load_dotenv
import os

load_dotenv("../../.env")

True

In [64]:
ls_client=Client()


In [65]:
dataset=ls_client.read_dataset(
    dataset_name="rag-evaluation-dataset"
)

In [66]:
dataset

Dataset(name='rag-evaluation-dataset', description='RAG evaluation dataset', data_type=<DataType.kv: 'kv'>, id=UUID('f496c5dd-ac70-495a-a544-c139bcd6f697'), created_at=datetime.datetime(2026, 7, 10, 21, 36, 29, 518272, tzinfo=TzInfo(0)), modified_at=datetime.datetime(2026, 7, 10, 21, 36, 29, 518272, tzinfo=TzInfo(0)), example_count=60, session_count=0, last_session_start_time=None, inputs_schema=None, outputs_schema=None, transformations=None, metadata={'runtime': {'sdk': 'langsmith-py', 'library': 'langsmith', 'runtime': 'python', 'platform': 'macOS-26.5.1-arm64-arm-64bit', 'sdk_version': '0.10.0', 'runtime_version': '3.12.13', 'langchain_version': '1.3.2', 'py_implementation': 'CPython', 'langchain_core_version': '1.4.9'}})

In [67]:
list(ls_client.list_examples(dataset_id=dataset.id,limit=50))

[<class 'langsmith.schemas.Example'>(id=936905f6-65df-448f-bf43-59b132ef9057, dataset_id=f496c5dd-ac70-495a-a544-c139bcd6f697, link='https://smith.langchain.com/o/0e808239-1007-49af-a944-8c47cd1c0199/datasets/f496c5dd-ac70-495a-a544-c139bcd6f697/e/936905f6-65df-448f-bf43-59b132ef9057'),
 <class 'langsmith.schemas.Example'>(id=2e2912a5-14d9-48d2-a8ed-6fba1aa0c499, dataset_id=f496c5dd-ac70-495a-a544-c139bcd6f697, link='https://smith.langchain.com/o/0e808239-1007-49af-a944-8c47cd1c0199/datasets/f496c5dd-ac70-495a-a544-c139bcd6f697/e/2e2912a5-14d9-48d2-a8ed-6fba1aa0c499'),
 <class 'langsmith.schemas.Example'>(id=621e575d-5ad3-497b-a018-129e7e78bcd4, dataset_id=f496c5dd-ac70-495a-a544-c139bcd6f697, link='https://smith.langchain.com/o/0e808239-1007-49af-a944-8c47cd1c0199/datasets/f496c5dd-ac70-495a-a544-c139bcd6f697/e/621e575d-5ad3-497b-a018-129e7e78bcd4'),
 <class 'langsmith.schemas.Example'>(id=c7c2dfbb-2ebb-480f-a6c5-2faae32cabd0, dataset_id=f496c5dd-ac70-495a-a544-c139bcd6f697, link='htt

In [68]:
list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[15].outputs

{'ground_truth': 'It has 7 USB ports, and each port has its own on/off switch.',
 'reference_context_ids': ['B0B7H3D2R9'],
 'reference_descriptions': ["Onfinio USB Hub 3.0, 7 Port USB Hub Splitter with Individual On/Off LED Switches, 5Gbps HighSpeed Data USB Extension for Laptop, iMac, USB Flash Drives, Mobile HDD, Printer, Camera and More [USB PORT HUB EXPANDER] 7 port usb hub which makes it very easy to connect Multi USB devices to your PC, laptop simultaneously, such as keyboard, mouse, card reader, mobile HDD, webcam, mp3 players, oculus, etc [5GBPS DATA TRANSFER SPEED] The USB 3.0 Hub extension provides up to 5 Gbps data transfer speed, which is more than 10 times faster than USB 2.0, and it is backwards compatible with USB 2.0/1.0 [INDIVIDUAL ON/OFF SWITCHES] USB hub has built-in on and off switches with led indicators for each port, being able to turn on or off the separate USB port without unplugging the equipment, blue LED lights let you know the working status easily. [WIDE C

In [69]:
list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[15].inputs

{'question': 'How many USB ports does this hub have and can I turn them off individually?'}

In [70]:
reference_input = list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[15].inputs
reference_output = list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[15].outputs

RAG pipeline

In [71]:
qdrant_client = QdrantClient(url="http://localhost:6333")


def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
   
    return response.data[0].embedding   


def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }


def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}    
"""

    return prompt




def generate_answer(prompt):

    response = openai.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning_effort="none"
    )
    
    return response.choices[0].message.content


def rag_pipeline(question, top_k=5):

    retrieved_context = retrieve_data(question, k=top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    final_answer= {

        "answer":answer,
        "question":question,
        "retrieved_context_ids":retrieved_context["retrieved_context_ids"],
        "retrieved_context":retrieved_context["retrieved_context"]
    }
    return final_answer

def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context

In [72]:
dataset

Dataset(name='rag-evaluation-dataset', description='RAG evaluation dataset', data_type=<DataType.kv: 'kv'>, id=UUID('f496c5dd-ac70-495a-a544-c139bcd6f697'), created_at=datetime.datetime(2026, 7, 10, 21, 36, 29, 518272, tzinfo=TzInfo(0)), modified_at=datetime.datetime(2026, 7, 10, 21, 36, 29, 518272, tzinfo=TzInfo(0)), example_count=60, session_count=0, last_session_start_time=None, inputs_schema=None, outputs_schema=None, transformations=None, metadata={'runtime': {'sdk': 'langsmith-py', 'library': 'langsmith', 'runtime': 'python', 'platform': 'macOS-26.5.1-arm64-arm-64bit', 'sdk_version': '0.10.0', 'runtime_version': '3.12.13', 'langchain_version': '1.3.2', 'py_implementation': 'CPython', 'langchain_core_version': '1.4.9'}})

In [73]:
dataset

Dataset(name='rag-evaluation-dataset', description='RAG evaluation dataset', data_type=<DataType.kv: 'kv'>, id=UUID('f496c5dd-ac70-495a-a544-c139bcd6f697'), created_at=datetime.datetime(2026, 7, 10, 21, 36, 29, 518272, tzinfo=TzInfo(0)), modified_at=datetime.datetime(2026, 7, 10, 21, 36, 29, 518272, tzinfo=TzInfo(0)), example_count=60, session_count=0, last_session_start_time=None, inputs_schema=None, outputs_schema=None, transformations=None, metadata={'runtime': {'sdk': 'langsmith-py', 'library': 'langsmith', 'runtime': 'python', 'platform': 'macOS-26.5.1-arm64-arm-64bit', 'sdk_version': '0.10.0', 'runtime_version': '3.12.13', 'langchain_version': '1.3.2', 'py_implementation': 'CPython', 'langchain_core_version': '1.4.9'}})

In [74]:
dataset

Dataset(name='rag-evaluation-dataset', description='RAG evaluation dataset', data_type=<DataType.kv: 'kv'>, id=UUID('f496c5dd-ac70-495a-a544-c139bcd6f697'), created_at=datetime.datetime(2026, 7, 10, 21, 36, 29, 518272, tzinfo=TzInfo(0)), modified_at=datetime.datetime(2026, 7, 10, 21, 36, 29, 518272, tzinfo=TzInfo(0)), example_count=60, session_count=0, last_session_start_time=None, inputs_schema=None, outputs_schema=None, transformations=None, metadata={'runtime': {'sdk': 'langsmith-py', 'library': 'langsmith', 'runtime': 'python', 'platform': 'macOS-26.5.1-arm64-arm-64bit', 'sdk_version': '0.10.0', 'runtime_version': '3.12.13', 'langchain_version': '1.3.2', 'py_implementation': 'CPython', 'langchain_core_version': '1.4.9'}})

In [75]:
rag_pipeline("can i get a charger")

{'answer': 'Yes. We have charging options in stock, including:\n\n1) BLACKSYNCZE iPhone Charger Cable (MFi Certified) – 2-pack, 10ft each, Lightning to USB (Grey). Fast charging up to 2.4A and MFi-certified compatibility with many iPhone models.\n\n2) LIQUIPEL Powertek Glow MFi Certified Lightning Cable – 5ft, Lightning to USB (Green). Fast charging and MFi certified.\n\n3) USB-C Cable (Fast Charging) – 3-pack, 10ft each. USB-A to USB-C, up to 3A max fast charging (for many USB-C devices like Samsung, LG, Moto, tablets, etc.).\n\n4) USB Outlet Extender Surge Protector with 3 USB ports (to plug your charger/cables into) – includes surge protection and smart USB charging ports.\n\nWhich device do you need to charge (iPhone with Lightning, or a USB-C phone), and do you want a 5ft or 10ft cable length?',
 'question': 'can i get a charger',
 'retrieved_context_ids': ['B0B63Z18FS',
  'B09V1V871J',
  'B0BGLJC82D',
  'B09SZ7222N',
  'B09ZVFHMJN'],
 'retrieved_context': ["BLACKSYNCZE iPhone Cha

RAGAS metrics

In [76]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy

/var/folders/_f/kw55hvzs0gn8bfjbbwvjg4mh0000gn/T/ipykernel_70293/3756680326.py:2: DeprecationWarning: Importing IDBasedContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextPrecision
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy
/var/folders/_f/kw55hvzs0gn8bfjbbwvjg4mh0000gn/T/ipykernel_70293/3756680326.py:2: DeprecationWarning: Importing IDBasedContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextRecall
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy
/var/folders/_f/kw55hvzs0gn8bfjbbwvjg4mh0000gn/T/ipykernel_70293/3756680326.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is depre

In [77]:
ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-5.4-mini"))
ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

/var/folders/_f/kw55hvzs0gn8bfjbbwvjg4mh0000gn/T/ipykernel_70293/840510326.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-5.4-mini"))
/var/folders/_f/kw55hvzs0gn8bfjbbwvjg4mh0000gn/T/ipykernel_70293/840510326.py:2: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))


In [78]:
reference_input

{'question': 'How many USB ports does this hub have and can I turn them off individually?'}

In [79]:
reference_output

{'ground_truth': 'It has 7 USB ports, and each port has its own on/off switch.',
 'reference_context_ids': ['B0B7H3D2R9'],
 'reference_descriptions': ["Onfinio USB Hub 3.0, 7 Port USB Hub Splitter with Individual On/Off LED Switches, 5Gbps HighSpeed Data USB Extension for Laptop, iMac, USB Flash Drives, Mobile HDD, Printer, Camera and More [USB PORT HUB EXPANDER] 7 port usb hub which makes it very easy to connect Multi USB devices to your PC, laptop simultaneously, such as keyboard, mouse, card reader, mobile HDD, webcam, mp3 players, oculus, etc [5GBPS DATA TRANSFER SPEED] The USB 3.0 Hub extension provides up to 5 Gbps data transfer speed, which is more than 10 times faster than USB 2.0, and it is backwards compatible with USB 2.0/1.0 [INDIVIDUAL ON/OFF SWITCHES] USB hub has built-in on and off switches with led indicators for each port, being able to turn on or off the separate USB port without unplugging the equipment, blue LED lights let you know the working status easily. [WIDE C

In [80]:
result=rag_pipeline(reference_input["question"])


In [81]:
result

{'answer': 'The Onfinio USB 3.0 hub has 7 USB ports, and yes—you can turn each port off individually using the built-in on/off switches with LED indicators.',
 'question': 'How many USB ports does this hub have and can I turn them off individually?',
 'retrieved_context_ids': ['B0B7H3D2R9',
  'B09ZVFHMJN',
  'B09SZ7222N',
  'B0BWHHJQQ9',
  'B0C6KM4H9M'],
 'retrieved_context': ["Onfinio USB Hub 3.0, 7 Port USB Hub Splitter with Individual On/Off LED Switches, 5Gbps HighSpeed Data USB Extension for Laptop, iMac, USB Flash Drives, Mobile HDD, Printer, Camera and More [USB PORT HUB EXPANDER] 7 port usb hub which makes it very easy to connect Multi USB devices to your PC, laptop simultaneously, such as keyboard, mouse, card reader, mobile HDD, webcam, mp3 players, oculus, etc [5GBPS DATA TRANSFER SPEED] The USB 3.0 Hub extension provides up to 5 Gbps data transfer speed, which is more than 10 times faster than USB 2.0, and it is backwards compatible with USB 2.0/1.0 [INDIVIDUAL ON/OFF SWITC

In [82]:
async def ragas_context_precision_id_based(run,example):

    sample=SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"]
    )
    
    scorer=IDBasedContextPrecision()
    return await scorer.single_turn_ascore(sample)

In [83]:
await ragas_context_precision_id_based(result,reference_output)

0.2

In [84]:
async def ragas_context_recall_id_based(run,example):

    sample=SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"]
    )
    
    scorer=IDBasedContextRecall()
    return await scorer.single_turn_ascore(sample)

In [86]:
await ragas_context_recall_id_based(result,reference_output)

1.0

In [87]:
async def ragas_faithfulness(run, example):

    sample = SingleTurnSample(
            user_input=run["question"],
            response=run["answer"],
            retrieved_contexts=run["retrieved_context"]
        )

    scorer = Faithfulness(llm=ragas_llm)
    
    return await scorer.single_turn_ascore(sample)

In [88]:
await ragas_faithfulness(result, reference_output)

1.0

In [89]:
async def ragas_relevancy(run, example):

    sample = SingleTurnSample(
        user_input=run["question"],
        response=run["answer"],
        retrieved_contexts=run["retrieved_context"]
    )

    scorer = ResponseRelevancy(llm=ragas_llm, embeddings=ragas_embeddings)

    return await scorer.single_turn_ascore(sample)

In [90]:
await ragas_relevancy(result, reference_output)

np.float64(0.8757979710906857)